In [ ]:
CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = ""  # set per notebook

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")

In [ ]:
import os, json

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/recently_played"
    if not os.path.exists(entity_path): return rows
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                for item in data.get("items", []):
                    if not item: continue
                    track = item.get("track", {})
                    artists = track.get("artists") or []
                    artist = artists[0] if artists else {}
                    rows.append({
                        "played_at": item.get("played_at"), "track_id": track.get("id"),
                        "track_name": track.get("name"), "artist_id": artist.get("id"),
                        "artist_name": artist.get("name"), "album_id": (track.get("album") or {}).get("id"),
                        "album_name": (track.get("album") or {}).get("name"),
                        "context_type": (item.get("context") or {}).get("type"),
                        "duration_ms": track.get("duration_ms"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows).dropDuplicates(["played_at", "track_id"])
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
